<script src="{{site.baseurl}}/assets/js/code-runner-analytics.js"></script>

## Popcorn Hack 1: Random SFI Record Number

`random.randint(1, record_count)` includes both endpoints, so every record from `1` through `4` is reachable. To show it, the runner picks 20 record numbers and counts how many times each came up.

In [ ]:
# CODE_RUNNER: Popcorn Hack 1 - Random SFI Record Number

import random

record_count = 4

# One random record number, inclusive of both 1 and record_count
selected = random.randint(1, record_count)
print("SFI record selected for QA:", selected)

# Repeat 20 times and count each result to show every valid ID can occur
counts = {1: 0, 2: 0, 3: 0, 4: 0}
for run in range(20):
    picked = random.randint(1, record_count)
    counts[picked] = counts[picked] + 1

print("Counts after 20 runs:", counts)
print("Any ID outside 1-4?", any(k < 1 or k > record_count for k in counts))

**Explanation:** The selected number is always 1, 2, 3, or 4, and never 0 or 5. Over 20 runs each ID usually appears, but random does **not** mean each shows up once before a repeat. Repeats are normal, and a count of 0 for one ID in a short run is possible (the more runs, the closer the counts get to even).

## Popcorn Hack 2: Random Structured SFI Car Part

`random.choice` selects one complete record (a dictionary) from the list, and then the fields are read by key.

In [ ]:
# CODE_RUNNER: Popcorn Hack 2 - Random SFI Part Record

import random

parts = [
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    }
]

selected_part = random.choice(parts)

print("Random SFI part selected for QA:")
print("  Product name:", selected_part["product_name"])
print("  Category:", selected_part["category"])
print("  Spec number:", selected_part["spec_number"])

**Explanation:** The output is always one whole record, so the name, category, and spec number always match each other (they never get mixed between records). The same record can be selected on consecutive runs.

## Popcorn Hack 3: Random SFI API Test

A random part and a random action are chosen together, then the action decides which backend-style response is printed. Five tests are run so different part/action combinations show up.

In [ ]:
# CODE_RUNNER: Popcorn Hack 3 - Random SFI API Test

import random

parts = [
    {"product_name": "Replacement Flywheels", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "spec_number": "2.1"}
]

actions = [
    "GET/search",
    "POST/create",
    "PUT/update"
]

for test_number in range(1, 6):
    part = random.choice(parts)
    action = random.choice(actions)
    print("Test", test_number, "-", action, "on spec", part["spec_number"])

    # Route each action to the matching backend-style response
    if action == "GET/search":
        print("   200 OK: search found", part["product_name"])
    elif action == "POST/create":
        print("   201 Created: validated and created", part["product_name"])
    else:
        print("   200 OK: updated", part["product_name"])

**Explanation:** Each run picks the part and the action independently, so combinations vary (a search on 1.1, then an update on 2.1, and so on). The `if`/`elif`/`else` chain makes sure every action gets exactly one matching response.

## Popcorn Hack 4 / Homework: SFI Backend QA Simulator

A reusable simulator: for each of `test_count` runs it picks a random part and a random action, then applies realistic backend rules, including **duplicate-spec handling** on create. `existing_spec_numbers` acts as the backend's stored data, so creates and removes change what later tests see.

In [ ]:
# CODE_RUNNER: Popcorn Hack 4 / Homework - SFI Backend QA Simulator

import random

parts = [
    {
        "product_name": "Replacement Flywheels",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    }
]

actions = [
    "GET/search",
    "POST/create",
    "PUT/update",
    "DELETE/remove"
]

existing_spec_numbers = ["1.1", "2.1"]
test_count = 5

for test_number in range(1, test_count + 1):
    part = random.choice(parts)
    action = random.choice(actions)
    spec = part["spec_number"]
    stored = spec in existing_spec_numbers

    print("Test " + str(test_number) + ": " + action + " | spec " + spec + " - " + part["product_name"])

    if action == "GET/search":
        if stored:
            print("   RESULT: found, returned the record")
        else:
            print("   RESULT: not found, no record with that spec")
    elif action == "POST/create":
        if stored:
            print("   RESULT: REJECTED, duplicate spec " + spec)
        else:
            existing_spec_numbers.append(spec)
            print("   RESULT: created, spec " + spec + " is now stored")
    elif action == "PUT/update":
        if stored:
            print("   RESULT: updated the stored record")
        else:
            print("   RESULT: cannot update, spec " + spec + " does not exist")
    else:
        if stored:
            existing_spec_numbers.remove(spec)
            print("   RESULT: removed, spec " + spec + " is no longer stored")
        else:
            print("   RESULT: nothing to remove, spec " + spec + " does not exist")

print()
print("Spec numbers in storage:", existing_spec_numbers)

**Explanation of the behavior:**

- **Random:** Every run picks a different part/action pairing, so the output differs each time (no `seed`, on purpose).
- **Duplicate-spec handling:** A `POST/create` on a spec already in `existing_spec_numbers` is rejected. If the spec is new, it is stored, so a later create of the same spec in the same run is now a duplicate.
- **State carries over:** A remove takes the spec out of storage, so a later search or update on it correctly fails with "not found" or "does not exist".
- **Every action has both outcomes:** each branch prints something whether or not the spec is stored, so no test is silent.

**Sample run** (yours will differ because the choices are random):

```
Test 1: POST/create | spec 1.2 - Multiple Disc Clutch Assemblies
   RESULT: created, spec 1.2 is now stored
Test 2: POST/create | spec 1.2 - Multiple Disc Clutch Assemblies
   RESULT: REJECTED, duplicate spec 1.2
...
Spec numbers in storage: ['1.1', '2.1', '1.2']
```